# Chapter 5: Rolling and Expanding Statistics

## 1. Introduction

Rolling and expanding statistics are useful for analyzing time series data. They help us understand trends, smooth short-term fluctuations, and measure variability over time.

- **Rolling statistics** calculate a statistic over a fixed-size moving window.
- **Expanding statistics** calculate a statistic over a window that starts at the first observation and grows as new observations arrive.

These techniques are commonly used in financial analysis, sensor monitoring, industrial process monitoring, and forecasting.

## 2. Rolling Statistics

### 2.1 Rolling Mean

A rolling mean calculates the average of observations within a moving window.

```python
df["rolling_mean"] = df["temperature"].rolling(window=3).mean()
```

Here, `window=3` means that each rolling mean is calculated using three consecutive observations.

For example, the first complete window contains `[100, 104, 102]`.

\[
\text{Mean} = \frac{100+104+102}{3} = 102
\]

The window then moves forward by one observation. The next window is `[104, 102, 110]`, whose mean is approximately `105.33`.

**Why are the first two values NaN?**

By default, pandas requires three valid observations to calculate a statistic with `window=3`. The first two positions do not contain enough observations.

### 2.2 The `min_periods` Parameter

The `min_periods` parameter specifies the minimum number of valid observations required to calculate a statistic.

```python
df["rolling_mean_1"] = (
    df["temperature"]
    .rolling(window=3, min_periods=1)
    .mean()
)
```

With `min_periods=1`, pandas calculates a mean as soon as one valid observation is available.

For the first three observations, the results are:

| Temperature | Rolling mean |
|---:|---:|
| 100 | 100.00 |
| 104 | 102.00 |
| 102 | 102.00 |

The window size remains three. `min_periods` changes the minimum number of observations required, not the maximum window size.

### 2.3 Rolling Standard Deviation

Standard deviation measures how spread out observations are around their mean.

```python
df["rolling_std"] = df["temperature"].rolling(window=3).std()
```

Pandas uses the **sample standard deviation** by default, with `ddof=1`.

\[
s = \sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}}
\]

For the window `[100, 104, 102]`, the mean is `102`.

\[
s = \sqrt{\frac{(-2)^2+2^2+0^2}{3-1}} = 2
\]

If every observation is identical, such as `[10, 10, 10]`, the standard deviation is `0` because there is no variation.

Standard deviation can help identify periods of relatively high or low variability in time series data.

### 2.4 Other Rolling Statistics

Pandas supports several useful rolling methods:

| Method | Purpose |
|---|---|
| `.mean()` | Calculates the rolling average |
| `.sum()` | Calculates the rolling sum |
| `.min()` | Finds the rolling minimum |
| `.max()` | Finds the rolling maximum |
| `.median()` | Calculates the rolling median |
| `.std()` | Calculates the rolling standard deviation |
| `.var()` | Calculates the rolling variance |

Example:

```python
df["rolling_min"] = df["temperature"].rolling(window=3).min()
df["rolling_max"] = df["temperature"].rolling(window=3).max()
df["rolling_median"] = df["temperature"].rolling(window=3).median()
```

Rolling statistics use only the observations in the current window. As the window moves, older observations leave and newer observations enter.

## 3. Expanding Statistics

### 3.1 Expanding Mean

An expanding window starts with the first observation and includes every subsequent observation. Unlike a rolling window, it never discards earlier observations.

```python
df["expanding_mean"] = df["temperature"].expanding().mean()
```

The expanding mean for the temperature series is:

| Number of observations | Observations included | Expanding mean |
|---:|---|---:|
| 1 | `[100]` | 100.00 |
| 2 | `[100, 104]` | 102.00 |
| 3 | `[100, 104, 102]` | 102.00 |
| 4 | `[100, 104, 102, 110]` | 104.00 |
| 5 | `[100, 104, 102, 110, 108]` | 104.80 |
| 6 | `[100, 104, 102, 110, 108, 112]` | 106.00 |

The expanding mean represents the average of all observations available up to each timestamp.

### 3.2 Expanding `min_periods`

The `min_periods` parameter also works with expanding windows.

```python
df["expanding_mean_3"] = (
    df["temperature"]
    .expanding(min_periods=3)
    .mean()
)
```

The first two values are `NaN` because at least three valid observations are required.

After the third observation, the window continues to grow. `min_periods=3` does not restrict the expanding window to three observations.

### 3.3 Other Expanding Statistics

Expanding windows support the same general statistical methods as rolling windows.

```python
df["expanding_std"] = df["temperature"].expanding().std()
df["expanding_min"] = df["temperature"].expanding().min()
df["expanding_max"] = df["temperature"].expanding().max()
df["expanding_sum"] = df["temperature"].expanding().sum()
```

Pandas uses sample standard deviation by default for expanding standard deviation as well. Therefore, the first result is `NaN` because a sample standard deviation requires at least two valid observations.

## 4. Rolling vs. Expanding Statistics

| Feature | Rolling | Expanding |
|---|---|---|
| Window size | Fixed | Increases over time |
| Earlier observations | Eventually leave the window | Remain included |
| Main purpose | Analyze recent behavior | Analyze cumulative history |
| Example | Average temperature over the last 3 hours | Average temperature since recording began |
| Sensitivity to recent changes | Usually more responsive | Can be less responsive as history grows |

### Practical Application in Chemical Engineering

Rolling statistics can be used to monitor recent reactor temperatures, pressure readings, viscosity measurements, or coating-process properties. They can help reveal short-term changes and fluctuations.

Expanding statistics can be used to track cumulative averages or other statistics throughout a production run or experimental study.

However, an expanding average may hide a recent change because older observations continue to influence the result. The choice of window should reflect the purpose of the analysis.

## 5. Code Summary

```python
# Rolling statistics
df["rolling_mean"] = df["temperature"].rolling(window=3).mean()
df["rolling_std"] = df["temperature"].rolling(window=3).std()
df["rolling_median"] = df["temperature"].rolling(window=3).median()
df["rolling_min"] = df["temperature"].rolling(window=3).min()
df["rolling_max"] = df["temperature"].rolling(window=3).max()

# Expanding statistics
df["expanding_mean"] = df["temperature"].expanding().mean()
df["expanding_std"] = df["temperature"].expanding().std()
df["expanding_min"] = df["temperature"].expanding().min()
df["expanding_max"] = df["temperature"].expanding().max()

# Display selected results
df[[
    "temperature",
    "rolling_mean",
    "rolling_std",
    "rolling_median",
    "expanding_mean",
    "expanding_std"
]]
```

**Important:** The examples assume that `df` contains the original temperature time series and that its index is the timestamp.


In [1]:
import pandas as pd

dates = pd.date_range(
    start="2026-01-01 09:00",
    periods=6,
    freq="h"
)

df = pd.DataFrame(
    {"temperature": [100, 104, 102, 110, 108, 112]},
    index=dates
)

df.index.name = "time"

df

,temperature
time,
2026-01-01 09:00:00,100
2026-01-01 10:00:00,104
2026-01-01 11:00:00,102
2026-01-01 12:00:00,110
2026-01-01 13:00:00,108
2026-01-01 14:00:00,112


In [2]:
df["rolling_mean_3"] = (df.temperature.rolling(window = 3).mean())

In [3]:
df

,temperature,rolling_mean_3
time,,
2026-01-01 09:00:00,100,NaN
2026-01-01 10:00:00,104,NaN
2026-01-01 11:00:00,102,102.000000
2026-01-01 12:00:00,110,105.333333
2026-01-01 13:00:00,108,106.666667
2026-01-01 14:00:00,112,110.000000


In [4]:
df["rolling_mean_3"] = (
    df["temperature"]
    .rolling(window=3, min_periods=1)
    .mean()
)
df

,temperature,rolling_mean_3
time,,
2026-01-01 09:00:00,100,100.000000
2026-01-01 10:00:00,104,102.000000
2026-01-01 11:00:00,102,102.000000
2026-01-01 12:00:00,110,105.333333
2026-01-01 13:00:00,108,106.666667
2026-01-01 14:00:00,112,110.000000


In [5]:
df["rolling_min"] = df["temperature"].rolling(window=3).min()
df["rolling_max"] = df["temperature"].rolling(window=3).max()

df[["temperature", "rolling_min", "rolling_max"]]

,temperature,rolling_min,rolling_max
time,,,
2026-01-01 09:00:00,100,NaN,NaN
2026-01-01 10:00:00,104,NaN,NaN
2026-01-01 11:00:00,102,100.0,104.0
2026-01-01 12:00:00,110,102.0,110.0
2026-01-01 13:00:00,108,102.0,110.0
2026-01-01 14:00:00,112,108.0,112.0


In [6]:
df["expanding_mean_3"] = (
    df["temperature"]
    .expanding(min_periods=3)
    .mean()
)
df

,temperature,rolling_mean_3,rolling_min,rolling_max,expanding_mean_3
time,,,,,
2026-01-01 09:00:00,100,100.000000,NaN,NaN,NaN
2026-01-01 10:00:00,104,102.000000,NaN,NaN,NaN
2026-01-01 11:00:00,102,102.000000,100.0,104.0,102.0
2026-01-01 12:00:00,110,105.333333,102.0,110.0,104.0
2026-01-01 13:00:00,108,106.666667,102.0,110.0,104.8
2026-01-01 14:00:00,112,110.000000,108.0,112.0,106.0


In [7]:
df["expanding_std"] = df["temperature"].expanding().std()
df

,temperature,rolling_mean_3,rolling_min,rolling_max,expanding_mean_3,expanding_std
time,,,,,,
2026-01-01 09:00:00,100,100.000000,NaN,NaN,NaN,NaN
2026-01-01 10:00:00,104,102.000000,NaN,NaN,NaN,2.828427
2026-01-01 11:00:00,102,102.000000,100.0,104.0,102.0,2.000000
2026-01-01 12:00:00,110,105.333333,102.0,110.0,104.0,4.320494
2026-01-01 13:00:00,108,106.666667,102.0,110.0,104.8,4.147288
2026-01-01 14:00:00,112,110.000000,108.0,112.0,106.0,4.732864


In [8]:
df["expanding_sum"] = df["temperature"].expanding().sum()
df

,temperature,rolling_mean_3,rolling_min,rolling_max,expanding_mean_3,expanding_std,expanding_sum
time,,,,,,,
2026-01-01 09:00:00,100,100.000000,NaN,NaN,NaN,NaN,100.0
2026-01-01 10:00:00,104,102.000000,NaN,NaN,NaN,2.828427,204.0
2026-01-01 11:00:00,102,102.000000,100.0,104.0,102.0,2.000000,306.0
2026-01-01 12:00:00,110,105.333333,102.0,110.0,104.0,4.320494,416.0
2026-01-01 13:00:00,108,106.666667,102.0,110.0,104.8,4.147288,524.0
2026-01-01 14:00:00,112,110.000000,108.0,112.0,106.0,4.732864,636.0


In [9]:
df["rolling_median"] = df["temperature"].rolling(window=3).median()
df["expanding_mean"] = df["temperature"].expanding(min_periods=3).mean()
df["expanding_std"] = df["temperature"].expanding(min_periods=3).std()
df[["temperature", "rolling_median","expanding_mean", "expanding_std"]]

,temperature,rolling_median,expanding_mean,expanding_std
time,,,,
2026-01-01 09:00:00,100,NaN,NaN,NaN
2026-01-01 10:00:00,104,NaN,NaN,NaN
2026-01-01 11:00:00,102,102.0,102.0,2.000000
2026-01-01 12:00:00,110,104.0,104.0,4.320494
2026-01-01 13:00:00,108,108.0,104.8,4.147288
2026-01-01 14:00:00,112,110.0,106.0,4.732864


# Chapter 5: Summary and Knowledge Check

## 1. Summary

In this chapter, we learned how to calculate rolling and expanding statistics using pandas.

### Key Takeaways

1. **Rolling windows** calculate statistics over a fixed number of observations and move forward through the series.
2. `rolling(window=3)` uses three observations per complete window.
3. The default `min_periods` for an integer rolling window is equal to the window size. Insufficient observations produce `NaN`.
4. `min_periods` specifies the minimum number of valid observations required, not the maximum window size.
5. Rolling mean, median, minimum, maximum, standard deviation, variance, and sum can help analyze recent time series behavior.
6. **Expanding windows** begin with the first observation and grow as new observations arrive.
7. `expanding().mean()` calculates the cumulative mean up to each timestamp.
8. `expanding(min_periods=3)` requires at least three valid observations before calculating a result, but the window continues to grow afterward.
9. Pandas uses sample standard deviation by default (`ddof=1`). A sample standard deviation requires at least two valid observations.
10. A window containing identical values, such as `[10, 10, 10]`, has a standard deviation of `0`.
11. Rolling statistics are useful for analyzing recent behavior, whereas expanding statistics describe cumulative behavior.

## 2. Knowledge-Check Questions

Try to answer these questions without referring to the lesson above.

### Conceptual Questions

1. What is the main difference between a rolling window and an expanding window?

2. Why does `rolling(window=3).mean()` return `NaN` for the first two observations by default?

3. What does `min_periods=1` change in a rolling calculation? Does it change the window size?

4. If a rolling window contains `[10, 10, 10]`, what is its standard deviation? Explain why.

5. Why does pandas use a denominator of \(n-1\) when calculating sample standard deviation by default?

6. What does `expanding(min_periods=3).mean()` return for the second observation, and why?

7. Why can an expanding mean be less responsive to sudden changes in recent observations?

8. Give one practical example of using rolling statistics and one example of using expanding statistics in chemical engineering.

### Coding Questions

Use the following dataset:

```python
import pandas as pd

dates = pd.date_range(
    start="2026-01-01 09:00",
    periods=6,
    freq="h"
)

df = pd.DataFrame(
    {"temperature": [100, 104, 102, 110, 108, 112]},
    index=dates
)

df.index.name = "time"
```

**Exercise 1: Rolling median**

Create a column named `rolling_median` using a window of three observations. What are the first three results?

**Exercise 2: Rolling maximum**

Create a column named `rolling_max` using a window of three observations. What is the value at `2026-01-01 12:00:00`?

**Exercise 3: Expanding mean**

Create a column named `expanding_mean` using an expanding window. Calculate the expected result at `2026-01-01 14:00:00` manually.

**Exercise 4: Minimum observations**

Calculate an expanding mean using `min_periods=3`. Explain why the first two results are `NaN`.

**Exercise 5: Expanding standard deviation**

Calculate the expanding standard deviation. Why is the first result `NaN`, and why is the second result also `NaN` when using `min_periods=3`?

**Exercise 6: Comparison**

At `2026-01-01 14:00:00`, calculate the mean of the last three observations and compare it with the expanding mean. Explain why the results differ.

## 3. Final Reflection

Before moving to the next chapter, make sure you can:

- Explain rolling and expanding windows in your own words.
- Predict when a rolling calculation will return `NaN`.
- Explain the purpose of `min_periods`.
- Calculate a rolling mean and an expanding mean manually.
- Explain the difference between population and sample standard deviation.
- Choose an appropriate statistic for monitoring recent behavior versus cumulative behavior.

**Completion criterion:** You should be able to write the relevant pandas code, predict its output, and explain why the output is correct without relying on memorized syntax alone.
